# Submissão 3 - Modelo End-to-End (DeBERTa v3 Fine-Tuning)

Este notebook implementa um classificador de texto baseado no **DeBERTa-v3-small**. 

### Objetivos e Otimizações:
1. **Fine-Tuning Integral**: O modelo Transformer é treinado em conjunto com a DNN final.
2. **Otimização de Memória (GPU 2GB)**: Uso de **Mixed Precision (FP16)** e **Gradient Accumulation** para processar o modelo numa MX330.
3. **Estabilidade**: Implementação de um **Linear Scheduler com Warmup** para preservar o conhecimento pré-treinado do modelo.
4. **Eficiência**: Tokenização otimizada para o tamanho real dos textos (128 tokens).

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os
import pickle
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, accuracy_score
from torch.cuda.amp import autocast, GradScaler

# Configuração de diretórios
DATA_DIR = "../data"
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

# 2. Confirmar que a GPU T4 está ativa
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"A usar dispositivo: {device}")


c:\Users\joaop\miniconda3\envs\AP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


A usar dispositivo: cpu


## 1. Carregamento e Mapeamento
Lemos os datasets de treino e validação e convertemos as categorias de texto para índices numéricos.

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, "dataset_treino.csv"))
df_val = pd.read_csv(os.path.join(DATA_DIR, "dataset_validacao.csv"))

# Remover qualquer linha que tenha texto ou label vazios antes de fazer mais nada
df_train = df_train.dropna(subset=['Text', 'Label'])
df_val = df_val.dropna(subset=['Text', 'Label'])

# Garantir que o texto é string
df_train['Text'] = df_train['Text'].astype(str)
df_val['Text'] = df_val['Text'].astype(str)

# 4. Criar o mapeamento das 5 classes
unique_labels = sorted(df_train['Label'].dropna().unique().tolist())
label_mapping = {k: v for v, k in enumerate(unique_labels)}
reverse_mapping = {v: k for k, v in label_mapping.items()}
print("Mapeamento do Projeto:", label_mapping)

df_train['label'] = df_train['Label'].map(label_mapping)
df_val['label'] = df_val['Label'].map(label_mapping)

Mapeamento concluído: {'anthropic': 0, 'google': 1, 'human': 2, 'meta': 3, 'openai': 4}


## 2. Tokenização Otimizada
Usamos o `max_length=128` porque os textos do projeto têm entre 100-120 caracteres. Isto poupa imensa VRAM.

In [ ]:
# 5. Tokenizador e Dataset
MODEL_NAME = 'microsoft/deberta-v3-small'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding='max_length', max_length=max_length)
        self.labels = labels.tolist()
        
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
        
    def __len__(self):
        return len(self.labels)

BATCH_SIZE = 16 

train_loader = DataLoader(TextDataset(df_train['Text'], df_train['label'], tokenizer), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TextDataset(df_val['Text'], df_val['label'], tokenizer), batch_size=BATCH_SIZE*2)
print("Dados prontos!")

## 3. Definição da Arquitetura
O modelo extrai a representação global do texto (Mean Pooling) e passa-a por uma camada densa.

In [ ]:
# 1. Construir o Modelo
class EndToEndDetector(nn.Module):
    def __init__(self, model_name, num_classes, hidden_size=128):
        super(EndToEndDetector, self).__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        
        for param in self.transformer.embeddings.parameters():
            param.requires_grad = False
        for layer in self.transformer.encoder.layer[:-2]:
            for param in layer.parameters():
                param.requires_grad = False
                
        self.fc1 = nn.Linear(self.transformer.config.hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask):
        out = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        mask_expanded = attention_mask.unsqueeze(-1).expand(out.last_hidden_state.size()).float()
        sum_embeddings = torch.sum(out.last_hidden_state * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask
        
        x = self.dropout(self.relu(self.fc1(mean_pooled)))
        return self.fc2(x)

num_classes = len(unique_labels)

model = EndToEndDetector(MODEL_NAME, num_classes=num_classes).to(device)

# 2. Configurar o Treino
EPOCHS = 3
LEARNING_RATE = 1e-5 

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=LEARNING_RATE, 
    weight_decay=0.01,
    eps=1e-6 
)

criterion = nn.CrossEntropyLoss()

total_steps = len(train_loader) * EPOCHS
num_warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=total_steps)

print("A iniciar o treino")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for i, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
            
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
            
        total_loss += loss.item()
        
        if (i+1) % 200 == 0:
            print(f"Época {epoch+1} | Batch {i+1}/{len(train_loader)} | Loss Média: {total_loss/(i+1):.4f}")
            
    print(f"✅ Época {epoch+1} Concluída - Média de Loss Final: {total_loss/len(train_loader):.4f}")

# Guardar os pesos
torch.save(model.state_dict(), 'subm3_deberta_final.pt')
print("Treino concluído e modelo guardado!")

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 9245.74it/s]
DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect

## 4. Treino com Otimizações de Memória e Scheduler
Aplicamos o que aprendemos na aula mas com truques para a GPU:
- **Gradient Accumulation**: Simulamos uma batch de 32 (8 steps * 4 batch).
- **Scheduler**: A taxa de aprendizagem diminui gradualmente para um ajuste fino.

In [ ]:
EPOCHS = 3
LEARNING_RATE = 2e-5
ACCUMULATION_STEPS = 8 # Simula batch de 32

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler() # Para Mixed Precision FP16

# Scheduler com Warmup (Aprendido na aula)
total_steps = (len(train_loader) // ACCUMULATION_STEPS) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)

print("Iniciando treino...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    optimizer.zero_grad() # Limpar no início da época
    
    for i, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Mixed Precision para poupar memória
        with autocast():
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss = loss / ACCUMULATION_STEPS # Normalizar para acumulação
            
        scaler.scale(loss).backward()
        
        # Atualizar pesos após acumular gradientes
        if (i + 1) % ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            scheduler.step() # Atualizar LR
            optimizer.zero_grad()
            
        total_loss += loss.item() * ACCUMULATION_STEPS
        
    print(f"Época {epoch+1} concluída. Loss média: {total_loss/len(train_loader):.4f}")

# Guardar pesos finais
torch.save(model.state_dict(), os.path.join(MODELS_DIR, 'subm3_deberta_endtoend.pt'))

##  735

C:\Users\joaop\AppData\Local\Temp\ipykernel_10684\3710126836.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() # Para Mixed Precision FP16
c:\Users\joaop\miniconda3\envs\AP\Lib\site-packages\torch\cuda\amp\grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
C:\Users\joaop\AppData\Local\Temp\ipykernel_10684\3710126836.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
c:\Users\joaop\miniconda3\envs\AP\Lib\site-packages\torch\cuda\amp\autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


Iniciando treino...


## 5. Avaliação Final
Validamos o modelo no conjunto de validação e exportamos os resultados para o formato da submissão.

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        with autocast():
            logits = model(input_ids, attention_mask)
        
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch['labels'].numpy())

print("\n=== Relatório de Avaliação ===")
print(classification_report(all_labels, all_preds, target_names=unique_labels))

# Exportar CSV para entrega
# df_val['predicted_source'] = [reverse_mapping[p] for p in all_preds]
# df_val.to_csv('../subm3/subm3_deberta_results.csv', index=False)